# 练习实验：Collaborative Filtering Recommender Systems

<img align="left" src="./images/movie_camera.png" style="width:40px;" alt="电影摄影机">

本练习将实现 Collaborative Filtering（协同过滤），构建电影推荐系统。

## 目录

<img src="./images/film_reel.png" style="width:40px;" alt="胶片卷">

- [1 - 符号说明](#1)
- [2 - Recommender Systems](#2)
- [3 - 电影评分数据集](#3)
- [4 - Collaborative Filtering 学习算法](#4)
  - [4.1 Collaborative Filtering Cost Function](#4.1)
    - [练习 1](#ex01)
- [5 - 学习电影推荐](#5)
- [6 - 生成推荐](#6)
- [7 - 恭喜完成](#7)


> **注意：** 为避免 autograder（自动评分器）报错，提交作业时请勿编辑或删除不参与评分的单元，也不要新增单元。
>
> **通过作业后**，如果希望尝试修改不参与评分的代码，可以按照本 Notebook 底部的说明操作。


## 软件包

<img align="left" src="./images/film_strip_vertical.png" style="width:40px;" alt="胶片">

继续使用已熟悉的 NumPy 和 TensorFlow。


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from recsys_utils import *

<a name="1"></a>
## 1 - 符号说明


| 符号 | 含义 | Python 名称 |
| --- | --- | --- |
| $r(i,j)$ | 标量；用户 $j$ 对电影 $i$ 评分时为 1，否则为 0 | — |
| $y(i,j)$ | 标量；用户 $j$ 对电影 $i$ 的评分，仅在 $r(i,j)=1$ 时有定义 | — |
| $\mathbf{w}^{(j)}$ | 用户 $j$ 的参数向量 | — |
| $b^{(j)}$ | 用户 $j$ 的 Bias 参数 | — |
| $\mathbf{x}^{(i)}$ | 电影 $i$ 的 Feature Vector | — |
| $n_u$ | 用户数量 | `num_users` |
| $n_m$ | 电影数量 | `num_movies` |
| $n$ | 特征数量 | `num_features` |
| $\mathbf{X}$ | 由各电影的 $\mathbf{x}^{(i)}$ 组成的矩阵 | `X` |
| $\mathbf{W}$ | 由各用户的 $\mathbf{w}^{(j)}$ 组成的矩阵 | `W` |
| $\mathbf{b}$ | 由 $b^{(j)}$ 组成的 Bias Vector | `b` |
| $\mathbf{R}$ | 元素为 $r(i,j)$ 的 Indicator Matrix | `R` |


<a name="2"></a>
## 2 - Recommender Systems

<img align="left" src="./images/film_rating.png" style="width:40px;" alt="电影评分">

本实验将 Collaborative Filtering 应用于电影评分数据。算法为每个用户学习一个表示其电影偏好的 Parameter Vector，同时为每部电影学习一个相同维度的 Feature Vector。两者的 Dot Product 加上 Bias，应能估计用户对该电影的评分。

下图展示这些向量的学习方式。


<figure>
<img src="./images/ColabFilterLearn.PNG" style="width:740px;height:250px;" alt="根据已有评分共同学习用户参数和电影特征">
</figure>


已有评分按矩阵保存：$Y$ 中的评分范围为 0.5–5，步长为 0.5；未评分的位置填 0。$R$ 在已评分的位置为 1。矩阵的行对应电影，列对应用户。

每个用户有参数向量 $\mathbf{w}^{(j)}$ 和 Bias，每部电影有特征向量 $\mathbf{x}^{(i)}$。利用已有用户—电影评分，同时学习这些参数。图中示例的目标是使 $\mathbf{w}^{(1)}\cdot\mathbf{x}^{(1)}+b^{(1)}$ 接近评分 4。

电影特征需要兼顾所有为它评分的用户，而用户参数需要兼顾该用户评分过的电影。这种共同学习的方式正是 Collaborative Filtering 名称的由来。


<figure>
<img src="./images/ColabFilterUse.PNG" style="width:640px;height:250px;" alt="使用已学习参数预测尚未评分的电影">
</figure>


学到电影特征与用户参数后，就能预测用户对未评分电影的可能评分。上图以用户 1 对电影 0 的预测为例。


本练习先实现 `cofi_cost_func`，计算 Collaborative Filtering 的 Objective Function，再使用 TensorFlow 的 Custom Training Loop 学习参数。先详细了解本实验的数据集与数据结构。


<a name="3"></a>
## 3 - 电影评分数据集

<img align="left" src="./images/film_rating.png" style="width:40px;" alt="电影评分">

数据来自 [MovieLens 的 ml-latest-small 数据集](https://grouplens.org/datasets/movielens/latest/)。相关文献：F. Maxwell Harper and Joseph A. Konstan, 2015, *The MovieLens Datasets: History and Context*, ACM Transactions on Interactive Intelligent Systems (TiiS), 5(4), 19:1–19:19，[DOI](https://doi.org/10.1145/2827872)。

课程使用的原始数据约包含 600 名用户对 9,000 部电影的评分。为缩小规模，本实验重点保留 2000 年以来的电影，得到 $n_u=443$ 名用户和 $n_m=4778$ 部电影。评分为 0.5–5，步长为 0.5。

下面加载 $Y$ 和 $R$。$Y$ 的形状为 $n_m\times n_u$，保存评分 $y^{(i,j)}$；$R$ 为相同形状的 Binary Indicator Matrix，用户 $j$ 为电影 $i$ 评分时 $R(i,j)=1$，否则为 0。

还会使用以下参数：

$$
\mathbf{X}=\begin{bmatrix}
(\mathbf{x}^{(0)})^T\\
(\mathbf{x}^{(1)})^T\\
\vdots\\
(\mathbf{x}^{(n_m-1)})^T
\end{bmatrix},\qquad
\mathbf{W}=\begin{bmatrix}
(\mathbf{w}^{(0)})^T\\
(\mathbf{w}^{(1)})^T\\
\vdots\\
(\mathbf{w}^{(n_u-1)})^T
\end{bmatrix},\qquad
\mathbf{b}=\begin{bmatrix}b^{(0)}&b^{(1)}&\cdots&b^{(n_u-1)}\end{bmatrix}.
$$

$\mathbf{X}$ 的第 $i$ 行对应电影 $i$ 的 Feature Vector，$\mathbf{W}$ 的第 $j$ 行对应用户 $j$ 的 Parameter Vector。两类向量都是 $n$ 维。本部分使用 $n=10$，因此 $\mathbf{X}$ 的形状为 $n_m\times10$，$\mathbf{W}$ 为 $n_u\times10$。代码中的 `b` 使用形状 `(1, num_users)` 的行向量，便于 Broadcasting。

先加载评分数据，并载入预先计算好的 $\mathbf{X}$、$\mathbf{W}$、$\mathbf{b}$，用于开发 Cost Function。后续才会通过训练学习这些参数。


In [ ]:
#Load data
X, W, b, num_movies, num_features, num_users = load_precalc_params_small()
Y, R = load_ratings_small()

print("Y", Y.shape, "R", R.shape)
print("X", X.shape)
print("W", W.shape)
print("b", b.shape)
print("num_features", num_features)
print("num_movies",   num_movies)
print("num_users",    num_users)

In [ ]:
#  From the matrix, we can compute statistics like average rating.
tsmean =  np.mean(Y[0, R[0, :].astype(bool)])
print(f"Average rating for movie 1 : {tsmean:0.3f} / 5" )

<a name="4"></a>
## 4 - Collaborative Filtering 学习算法

<img align="left" src="./images/film_filter.png" style="width:40px;" alt="电影筛选">

首先实现 Objective Function。对于电影推荐，模型包含电影的 $n$ 维特征向量 $\mathbf{x}^{(0)},\ldots,\mathbf{x}^{(n_m-1)}$、用户参数向量 $\mathbf{w}^{(0)},\ldots,\mathbf{w}^{(n_u-1)}$，以及用户 Bias $b^{(0)},\ldots,b^{(n_u-1)}$。

用户 $j$ 对电影 $i$ 的预测评分为 $\hat y^{(i,j)}=\mathbf{w}^{(j)}\cdot\mathbf{x}^{(i)}+b^{(j)}$。目标是根据已有评分学习这些参数，使 Squared Error 尽可能小。

下面完成 `cofi_cost_func`，计算 Collaborative Filtering 的 Cost。


<a name="4.1"></a>
### 4.1 Collaborative Filtering Cost Function

Cost Function 为：

$$
\begin{aligned}
J(\mathbf{X},\mathbf{W},\mathbf{b})
&=\frac12\sum_{(i,j):r(i,j)=1}\left(\mathbf{w}^{(j)}\cdot\mathbf{x}^{(i)}+b^{(j)}-y^{(i,j)}\right)^2\\
&\quad+\underbrace{\frac{\lambda}{2}\sum_{j=0}^{n_u-1}\sum_{k=0}^{n-1}(w_k^{(j)})^2+\frac{\lambda}{2}\sum_{i=0}^{n_m-1}\sum_{k=0}^{n-1}(x_k^{(i)})^2}_{\text{Regularization}}.
\end{aligned}\tag{1}
$$

第一项只对 $r(i,j)=1$ 的用户—电影组合求和，也可写成：

$$J=\frac12\sum_{j=0}^{n_u-1}\sum_{i=0}^{n_m-1}r(i,j)\left(\mathbf{w}^{(j)}\cdot\mathbf{x}^{(i)}+b^{(j)}-y^{(i,j)}\right)^2+\text{Regularization}.$$

请实现 `cofi_cost_func`，返回这个 Cost。


<a name="ex01"></a>
### 练习 1

**使用 for 循环实现**

先实现不含 Regularization 的 Cost Function，并运行对应测试；正确后，再添加 Regularization 并运行相关测试。只有当 `R[i,j]==1` 时，才应累计用户 $j$ 对电影 $i$ 的评分误差。


In [ ]:
# GRADED FUNCTION: cofi_cost_func
# UNQ_C1

def cofi_cost_func(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    nm, nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    
        
        
        
            
            
            
            
    
    
    ### END CODE HERE ### 

    return J

<details>
<summary><strong>点击查看提示</strong></summary>

使用两个 `for` 循环对应式 (1) 的求和，先完成不含 Regularization 的版本。向量运算可使用 `np.dot()`，平方可使用 `np.square()`。注意区分索引 `i` 与 `j`，最后不要忘记除以 2。

```python
### START CODE HERE ###  
for j in range(nu):


    for i in range(nm):


### END CODE HERE ###
```

<details>
<summary><strong>点击查看更多提示</strong></summary>

下面先从矩阵中取出所需元素再计算，也可以直接引用矩阵元素。此代码尚不包含 Regularization。

```python
nm,nu = Y.shape
J = 0
### START CODE HERE ###  
for j in range(nu):
    w = W[j,:]
    b_j = b[0,j]
    for i in range(nm):
        x = 
        y = 
        r =
        J += 
J = J/2
### END CODE HERE ###
```

<details>
<summary><strong>查看完整的无 Regularization 实现</strong></summary>

```python
nm,nu = Y.shape
J = 0
### START CODE HERE ###  
for j in range(nu):
    w = W[j,:]
    b_j = b[0,j]
    for i in range(nm):
        x = X[i,:]
        y = Y[i,j]
        r = R[i,j]
        J += np.square(r * (np.dot(w,x) + b_j - y ) )
J = J/2
### END CODE HERE ###
```

<details>
<summary><strong>Regularization 提示</strong></summary>

将 `W` 和 `X` 中的所有元素分别平方后求和，再乘以相应系数。可使用 `np.square()` 和 `np.sum()`。

<details>
<summary><strong>Regularization 的详细实现</strong></summary>

```python
J += (lambda_/2) * (np.sum(np.square(W)) + np.sum(np.square(X)))
```

</details>
</details>
</details>
</details>
</details>


In [ ]:
# Reduce the data set size so that this runs faster
num_users_r = 4
num_movies_r = 5 
num_features_r = 3

X_r = X[:num_movies_r, :num_features_r]
W_r = W[:num_users_r,  :num_features_r]
b_r = b[0, :num_users_r].reshape(1,-1)
Y_r = Y[:num_movies_r, :num_users_r]
R_r = R[:num_movies_r, :num_users_r]

# Evaluate cost function
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

**预期输出（lambda = 0）：** `13.67`。


In [ ]:
# Evaluate cost function with regularization 
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

**预期输出：** `28.09`。


In [ ]:
# Public tests
from public_tests import *
test_cofi_cost_func(cofi_cost_func)

**Vectorization 实现**

优化过程中会反复计算 $J$，因此采用 Vectorization 很有价值。所需 Linear Algebra 并非本课程重点，下面直接提供实现。熟悉 Linear Algebra 的读者也可以先自行尝试。

运行下面的代码，确认结果与循环实现一致。


In [ ]:
def cofi_cost_func_v(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Vectorized for speed. Uses tensorflow operations to be compatible with custom training loop.
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y)*R
    J = 0.5 * tf.reduce_sum(j**2) + (lambda_/2) * (tf.reduce_sum(X**2) + tf.reduce_sum(W**2))
    return J

In [ ]:
# Evaluate cost function
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

# Evaluate cost function with regularization 
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

**预期输出**

```text
Cost: 13.67
Cost (with regularization): 28.09
```


<a name="5"></a>
## 5 - 学习电影推荐

<img align="left" src="./images/film_man_action.png" style="width:40px;" alt="电影场记板">

完成 Cost Function 后，可以训练算法，为自己生成电影推荐。在下面的单元中填写个人电影评分。代码已提供一些示例偏好；先用它们确认流程正常，再改成自己的评分。全部电影可在[电影列表](data/small_movie_list.csv)中查看。


In [ ]:
movieList, movieList_df = load_Movie_List_pd()

my_ratings = np.zeros(num_movies)          #  Initialize my ratings

# Check the file small_movie_list.csv for id of each movie in our dataset
# For example, Toy Story 3 (2010) has ID 2700, so to rate it "5", you can set
my_ratings[2700] = 5 

#Or suppose you did not enjoy Persuasion (2007), you can set
my_ratings[2609] = 2;

# We have selected a few movies we liked / did not like and the ratings we
# gave are as follows:
my_ratings[929]  = 5   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 3   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003)
my_rated = [i for i in range(len(my_ratings)) if my_ratings[i] > 0]

print('\nNew user ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0 :
        print(f'Rated {my_ratings[i]} for  {movieList_df.loc[i,"title"]}');

将这些评分加入 $Y$ 和 $R$，并对评分进行 Normalization。


In [ ]:
# Reload ratings
Y, R = load_ratings_small()

# Add new user ratings to Y 
Y = np.c_[my_ratings, Y]

# Add new user indicator matrix to R
R = np.c_[(my_ratings != 0).astype(int), R]

# Normalize the Dataset
Ynorm, Ymean = normalizeRatings(Y, R)

准备训练模型：初始化参数，并选择 Adam Optimizer。


In [ ]:
#  Useful Values
num_movies, num_users = Y.shape
num_features = 100

# Set Initial Parameters (W, X), use tf.Variable to track these variables
tf.random.set_seed(1234) # for consistent results
W = tf.Variable(tf.random.normal((num_users,  num_features),dtype=tf.float64),  name='W')
X = tf.Variable(tf.random.normal((num_movies, num_features),dtype=tf.float64),  name='X')
b = tf.Variable(tf.random.normal((1,          num_users),   dtype=tf.float64),  name='b')

# Instantiate an optimizer.
optimizer = keras.optimizers.Adam(learning_rate=1e-1)

下面训练 Collaborative Filtering 模型，学习参数 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$。


同时学习 $w$、$b$、$x$ 的计算不直接对应本课程此前使用的标准 Neural Network Layers，因此这里采用 Custom Training Loop。

回顾 Gradient Descent，每轮执行：计算 Forward Pass；计算 Loss 对参数的导数；利用 Learning Rate 和导数更新参数，重复直到满足结束条件。

TensorFlow 可以自动计算导数。在 `tf.GradientTape()` 上下文中，对 TensorFlow Variables 的相关运算会被记录；随后调用 `tape.gradient()`，得到 Loss 对这些变量的 Gradients，再通过 Optimizer 更新参数。

这只是对 Automatic Differentiation 和 Custom Training Loop 的简要介绍，其他 Machine Learning 框架也提供类似功能，可进一步查阅各框架的相关说明。


In [ ]:
iterations = 200
lambda_ = 1
for iter in range(iterations):
    # Use TensorFlow’s GradientTape
    # to record the operations used to compute the cost 
    with tf.GradientTape() as tape:

        # Compute the cost (forward pass included in cost)
        cost_value = cofi_cost_func_v(X, W, b, Ynorm, R, lambda_)

    # Use the gradient tape to automatically retrieve
    # the gradients of the trainable variables with respect to the loss
    grads = tape.gradient( cost_value, [X,W,b] )

    # Run one step of gradient descent by updating
    # the value of the variables to minimize the loss.
    optimizer.apply_gradients( zip(grads, [X,W,b]) )

    # Log periodically.
    if iter % 20 == 0:
        print(f"Training loss at iteration {iter}: {cost_value:0.1f}")

<a name="6"></a>
## 6 - 生成推荐

下面计算所有用户对所有电影的预测评分，并展示根据 `my_ratings` 生成的推荐。用户 $j$ 对电影 $i$ 的模型输出为 $\mathbf{w}^{(j)}\cdot\mathbf{x}^{(i)}+b^{(j)}$，可通过 Matrix Multiplication 一次性计算全部组合。由于训练时对评分做了 Mean Normalization，恢复原评分尺度时还需加回电影的均值。


In [ ]:
# Make a prediction using trained weights and biases
p = np.matmul(X.numpy(), np.transpose(W.numpy())) + b.numpy()

#restore the mean
pm = p + Ymean

my_predictions = pm[:,0]

# sort predictions
ix = tf.argsort(my_predictions, direction='DESCENDING')

for i in range(17):
    j = ix[i]
    if j not in my_rated:
        print(f'Predicting rating {my_predictions[j]:0.2f} for movie {movieList[j]}')

print('\n\nOriginal vs Predicted ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Original {my_ratings[i]}, Predicted {my_predictions[i]:0.2f} for {movieList[i]}')

实际应用中可以结合额外信息改善推荐。上面排名前几百的电影，其预测评分差异较小。因此，还可以从这些候选电影中选择平均评分较高、且评分人数超过 20 的电影。本节使用 [Pandas](https://pandas.pydata.org/) DataFrame 提供的排序功能。


In [ ]:
filter=(movieList_df["number of ratings"] > 20)
movieList_df["pred"] = my_predictions
movieList_df = movieList_df.reindex(columns=["pred", "mean rating", "number of ratings", "title"])
movieList_df.loc[ix[:300]].loc[filter].sort_values("mean rating", ascending=False)

<a name="7"></a>
## 7 - 恭喜完成！

<img align="left" src="./images/film_award.png" style="width:40px;" alt="电影奖杯">

你已经实现了一个实用的 Recommender System！


<details>
<summary><span style="color:darkgreen;"><strong>如需尝试修改不参与评分的代码，请点击这里。</strong></span></summary>

**重要提示：请在通过作业后再执行这些操作，以免影响 autograder。**

1. 在 Notebook 菜单中，依次选择 `View` > `Cell Toolbar` > `Edit Metadata`。
2. 在希望锁定或解锁的代码单元旁，点击 `Edit Metadata`。
3. 将 `editable` 属性设为 `true` 以解锁，或设为 `false` 以锁定。
4. 在 Notebook 菜单中，依次选择 `View` > `Cell Toolbar` > `None`。

下面的简短演示展示了上述步骤：

<img src="https://lh3.google.com/u/0/d/14Xy_Mb17CZVgzVAgq7NCjMVBvSae3xO1" alt="解锁 Notebook 单元的操作演示（unlock_cells.gif）">

</details>
